# 🎙️ OMNIVOICE — STUDIO LỒNG TIẾNG REVIEW PHIM ĐA NHÂN VẬT (CHUẨN SRT M1-M10, F1-F10)

### 🔒 ĐỘC QUYỀN: HỆ THỐNG KHÓA GIỌNG CHUẨN VĨNH VIỄN & BỘ NHỚ ĐỆM DRIVE/KAGGLE

*Tương thích 100% định dạng phụ đề SRT của các Tool Review Phim / Truyện / Anime*

*Mã nguồn gốc: [k2-fsa/OmniVoice](https://github.com/k2-fsa/OmniVoice)*



---



### ⚡ TÍNH NĂNG NỔI BẬT ĐÃ ĐƯỢC TỐI ƯU HÓA:

1. **Tự động nhận diện môi trường (Colab / Kaggle / Local)**:

   - Trên Colab: Tự động kết nối Google Drive lưu Model & Giọng vĩnh viễn (không phải tải lại 4-5GB).

   - Trên Kaggle: Tự động lưu cache tại `/kaggle/working` không lo lỗi phân quyền.

2. **Khóa giọng chuẩn 100% (Deterministic Voice Anchor)**: Giữ nguyên chất giọng `M1`, `M2`, `F1`... từ đầu đến cuối phim, không bị đổi giọng ngẫu nhiên.

3. **Tự động làm sạch từ khóa (Sanitize Whitelist)**: Không bao giờ bị lỗi `Unsupported instruct items`.

4. **Giao diện Web UI chia 2 bước rõ ràng**: Chạy ổn định 100%, không bị văng lỗi hay xung đột giao diện!



---



### 🚀 HƯỚNG DẪN CHẠY NHANH:

- **Cách 1 (Nhanh nhất)**: Bấm menu **Thời gian chạy (Runtime)** -> Chọn **Chạy tất cả (Run all)** (phím tắt `Ctrl + F9`).

- **Cách 2**: Bấm lần lượt nút **Play (▶)** ở ô `[1/2]` rồi ô `[2/2]` bên dưới.



In [ ]:
# @title ⚙️ [BƯỚC 1/2] KIỂM TRA GPU, MÔI TRƯỜNG & CÀI ĐẶT THƯ VIỆN { display-mode: "form" }
# @markdown Bấm nút Play để cài đặt môi trường:
LUU_VAO_GOOGLE_DRIVE = True #@param {type:"boolean"}

import warnings
warnings.filterwarnings("ignore")
import os, sys, time, re, shutil, zipfile, json
import numpy as np

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

print("="*65)
print("⏳ [1/2] ĐANG THIẾT LẬP MÔI TRƯỜNG & BỘ NHỚ ĐỆM...")
print("="*65)

# Tự động nhận diện môi trường: Google Colab, Kaggle hay Local
IS_COLAB = False
IS_KAGGLE = os.path.exists("/kaggle")
try:
    import google.colab
    IS_COLAB = True
except Exception:
    pass

if IS_COLAB and LUU_VAO_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        print("📁 Đang kết nối Google Drive (Lưu vĩnh viễn Model & Giọng nhân vật)...")
        drive.mount('/content/drive')
        BASE_DIR = "/content/drive/MyDrive/OmniVoice_Studio"
        print(f"✅ Đã kết nối Google Drive: {BASE_DIR}")
    except Exception as e:
        print(f"⚠️ Không dùng được Google Drive ({e}). Sử dụng thư mục tạm Colab.")
        BASE_DIR = "/content/OmniVoice_Studio"
elif IS_KAGGLE:
    BASE_DIR = "/kaggle/working/OmniVoice_Studio"
    print(f"✅ Đang chạy trên Kaggle! Thư mục làm việc: {BASE_DIR}")
else:
    BASE_DIR = os.path.abspath("./OmniVoice_Studio")
    print(f"✅ Đang chạy trên máy cục bộ (Local): {BASE_DIR}")

HF_CACHE_DIR = os.path.join(BASE_DIR, "hf_cache")
TORCH_CACHE_DIR = os.path.join(BASE_DIR, "torch_cache")
PIP_CACHE_DIR = os.path.join(BASE_DIR, "pip_cache")
CACHE_DIR = os.path.join(BASE_DIR, "character_voices")

for d in [HF_CACHE_DIR, TORCH_CACHE_DIR, PIP_CACHE_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["TORCH_HOME"] = TORCH_CACHE_DIR

# Kiểm tra GPU
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"🚀 GPU Sẵn Sàng: {gpu_name} ({vram:.1f} GB VRAM)")
else:
    print("⚠️ CẢNH BÁO: Đang dùng CPU! Tốc độ sẽ chậm. Hãy bật GPU trong cài đặt Runtime.")

# Cài đặt thư viện
pip_cache_opt = f'--cache-dir "{PIP_CACHE_DIR}"' if os.path.exists(PIP_CACHE_DIR) else ''
!pip install {pip_cache_opt} -q "omnivoice>=0.2.1" soundfile accelerate pydub gradio pyngrok
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared 2>/dev/null || true

print("\n✅ [BƯỚC 1/2] Cài đặt môi trường thành công! Tiếp tục chạy ô [BƯỚC 2/2] bên dưới.")


In [ ]:
# @title 🚀 [BƯỚC 2/2] NẠP MODEL OMNIVOICE & KHỞI CHẠY WEB UI STUDIO { display-mode: "form" }
# @markdown Bấm nút Play hoặc nhấn phím tắt **Ctrl + F9** để chạy tất cả.
# @markdown Mọi thông số Token & Tên miền cố định **ĐÃ ĐƯỢC LƯU SẴN 100% TRONG FILE**, BÁC KHÔNG CẦN TÌM HAY SỬA GÌ NỮA!
NGROK_AUTHTOKEN = "3JYfiFRrXHoFYeMq5hh219lbnyR_2VF5LDb2AJDpiSo1SefAm" #@param {type:"string"}
NGROK_DOMAIN = "upturned-evict-geologic.ngrok-free.dev" #@param {type:"string"}
LOAD_WHISPER_ASR = True #@param {type:"boolean"}

import warnings
warnings.filterwarnings("ignore")

import os, sys, time, re, shutil, zipfile, tempfile, json, base64, traceback
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

print("⏳ [1/3] Đang khởi động hệ thống & nạp thư viện âm thanh...", flush=True)

# Kích hoạt cơ chế tự động giữ kết nối chống ngủ gật (Anti-Disconnect / Anti-Idle) trên Google Colab
try:
    from IPython.display import display, HTML
    display(HTML('''
    <script>
    function ColabAntiIdle(){
        console.log("⚡ [OmniVoice] Anti-Idle Ping:", new Date().toLocaleTimeString());
        let btn = document.querySelector("colab-connect-button") || document.querySelector("#connect");
        if(btn) btn.click();
    }
    if(!window._omnivoice_idle_timer){
        window._omnivoice_idle_timer = setInterval(ColabAntiIdle, 60000);
        console.log("✅ Đã kích hoạt cơ chế chống ngắt kết nối Google Colab!");
    }
    </script>
    '''))
except Exception:
    pass

from pydub import AudioSegment
import numpy as np
import torch
import soundfile as sf

# Tự động phát hiện và tự cài đặt thư viện nếu người dùng quên chạy Bước 1 hoặc vừa đổi GPU Kaggle
try:
    import gradio as gr
    from omnivoice import OmniVoice, OmniVoiceGenerationConfig, VoiceClonePrompt
except ImportError:
    print("⏳ Phát hiện thiếu thư viện (do vừa khởi động lại Kernel / bật GPU). Đang tự động cài đặt nhanh...")
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "omnivoice>=0.2.1", "soundfile", "accelerate", "pydub", "gradio"], check=True)
    import gradio as gr
    from omnivoice import OmniVoice, OmniVoiceGenerationConfig, VoiceClonePrompt
    print("✅ Đã tự động cài đặt xong thư viện hoàn tất!")

# Tự động nhận diện và đảm bảo BASE_DIR / CACHE_DIR luôn chuẩn xác trên Colab, Kaggle hoặc Local
if os.path.exists("/kaggle"):
    BASE_DIR = "/kaggle/working/OmniVoice_Studio"
elif os.path.exists("/content/drive/MyDrive"):
    BASE_DIR = "/content/drive/MyDrive/OmniVoice_Studio"
elif os.path.exists("/content"):
    BASE_DIR = "/content/OmniVoice_Studio"
else:
    BASE_DIR = os.path.abspath("./OmniVoice_Studio")

CACHE_DIR = os.path.join(BASE_DIR, "character_voices")

for p in [BASE_DIR, CACHE_DIR]:
    os.makedirs(p, exist_ok=True)

os.environ["HF_HOME"] = os.path.join(BASE_DIR, "hf_cache")
os.environ["TORCH_HOME"] = os.path.join(BASE_DIR, "torch_cache")

# Tự động nạp file giọng .pt từ bất kỳ Kaggle Dataset nào được kết nối với notebook
if os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f.endswith(".pt") or f == "character_config.json":
                try:
                    dest_file = os.path.join(CACHE_DIR, f)
                    if not os.path.exists(dest_file):
                        shutil.copy(os.path.join(root, f), dest_file)
                        print(f"📦 Đã tự động nạp file {f} từ Kaggle Dataset!")
                except Exception:
                    pass

# Giải mã và bung các file giọng Clone được nhúng trực tiếp trong code EMBEDDED_CLONE_VOICES
def unpack_embedded_clones():
    if "EMBEDDED_CLONE_VOICES" in globals() and isinstance(EMBEDDED_CLONE_VOICES, dict):
        for tag_b64, b64_str in EMBEDDED_CLONE_VOICES.items():
            if b64_str and isinstance(b64_str, str) and len(b64_str.strip()) > 10:
                dest_pt = os.path.join(CACHE_DIR, f"{tag_b64.upper().strip()}.pt")
                try:
                    with open(dest_pt, "wb") as f:
                        f.write(base64.b64decode(b64_str.strip()))
                    print(f"✅ Đã nạp thành công giọng clone [{tag_b64.upper().strip()}] trực tiếp từ Code Notebook!")
                except Exception as e:
                    print(f"⚠️ Lỗi giải mã giọng clone [{tag_b64}] từ Code: {e}")

print("="*65)
print("⏳ [2/3] ĐANG NẠP MÔ HÌNH OMNIVOICE VÀO GPU (Vui lòng đợi 30-60 giây)...", flush=True)
if os.path.exists(os.path.join(os.environ.get("HF_HOME", ""), "hub")):
    print("⚡ Đã có sẵn model trong bộ nhớ đệm! Nạp trực tiếp trong vài giây...")
else:
    print("⏳ Lần đầu tiên: Đang tải model ~2.4GB về lưu vào bộ nhớ đệm...")
print("="*65 + "\n")

device = "cuda:0" if torch.cuda.is_available() else "cpu"
model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    cache_dir=os.environ.get("HF_HOME"),
    device_map=device,
    dtype=torch.float16 if device != "cpu" else torch.float32,
    load_asr=LOAD_WHISPER_ASR,
)
if hasattr(model, "eval"):
    model.eval()

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ"
gpu_info_banner = f"🚀 **Phần cứng đang chạy:** GPU `{gpu_name}` (Tốc độ tối đa)" if torch.cuda.is_available() else "⚠️ **CẢNH BÁO: ĐANG CHẠY TRÊN CPU!** Tốc độ sẽ rất chậm. Trên Kaggle/Colab, hãy vào menu **Settings (hoặc Runtime) -> Accelerator -> Chọn GPU T4 / P100** để nhanh gấp 15 lần!"
print(f"✅ Nạp mô hình OmniVoice thành công! ({device})")
if not torch.cuda.is_available():
    print("⚠️ CẢNH BÁO: Chưa kích hoạt GPU! Hãy bật GPU trong cài đặt Kaggle/Colab.")

# Whitelist các từ khóa instruct hợp lệ tuyệt đối của OmniVoice
VALID_INSTRUCT_ITEMS = {
    "american accent", "australian accent", "british accent", "canadian accent",
    "child", "chinese accent", "elderly", "female", "high pitch", "indian accent",
    "japanese accent", "korean accent", "low pitch", "male", "middle-aged",
    "moderate pitch", "portuguese accent", "russian accent", "teenager",
    "very high pitch", "very low pitch", "whisper", "young adult"
}

def sanitize_instruct(instruct_str, default="male, young adult, moderate pitch"):
    """Tự động lọc sạch mọi từ khóa ngoài whitelist để chống lỗi Unsupported"""
    if not instruct_str or not str(instruct_str).strip():
        return default
    items = [x.strip().lower() for x in str(instruct_str).split(",") if x.strip()]
    valid = [x for x in items if x in VALID_INSTRUCT_ITEMS]
    if not valid:
        return default
    return ", ".join(valid)

# ==============================================================================
# 🎯 CẤU HÌNH SEED & MÔ TẢ GIỌNG GỐC (LƯU CỨNG VĨNH VIỄN TRỰC TIẾP TRONG CODE NOTEBOOK)
# Bạn có thể chỉnh sửa trực tiếp các con số seed ở đây (ví dụ M1: 15161).
# Khi lưu file notebook trên Kaggle/Colab, các con số này sẽ không bao giờ bị mất!
# ==============================================================================
CHARACTER_DEFAULTS = {
    # === NHÂN VẬT NAM (M1 - M10) ===
    "M1": {"name": "M1 (Nam chính - Trầm ấm)", "seed": 15161, "instruct": "male, young adult, low pitch", "desc": "Nam trẻ trầm ấm, nam tính, phong thái anh hùng"},
    "M2": {"name": "M2 (Nam phụ / Trùm / Sếp)", "seed": 1002, "instruct": "male, middle-aged, very low pitch", "desc": "Nam trung niên cực trầm uy lực, lạnh lùng"},
    "M3": {"name": "M3 (Lão gia / Sư phụ)", "seed": 1003, "instruct": "male, elderly, low pitch", "desc": "Ông già trầm khàn, già dặn, thông thái"},
    "M4": {"name": "M4 (Nam thiếu niên / Thiếu gia)", "seed": 1004, "instruct": "male, teenager, moderate pitch", "desc": "Nam trẻ thanh thoát, lanh lợi, nhiệt huyết"},
    "M5": {"name": "M5 (Giang hồ / Hổ báo)", "seed": 1005, "instruct": "male, middle-aged, low pitch", "desc": "Nam trung niên gằn giọng, thô ráp"},
    "M6": {"name": "M6 (Thư sinh / Tri thức)", "seed": 1006, "instruct": "male, young adult, moderate pitch", "desc": "Nam trẻ nhẹ nhàng, điềm đạm, trí thức"},
    "M7": {"name": "M7 (Hài hước / Lém lỉnh)", "seed": 1007, "instruct": "male, young adult, energetic, high pitch", "desc": "Nam sôi nổi, vui tươi, lém lỉnh"},
    "M8": {"name": "M8 (Chiến binh / Dũng tướng)", "seed": 1008, "instruct": "male, adult, strong, resonant", "desc": "Nam dũng mãnh, giọng vang, hào sảng"},
    "M9": {"name": "M9 (Quái kiệt / Dị nhân)", "seed": 1009, "instruct": "male, middle-aged, raspy, eccentric", "desc": "Nam giọng khàn đặc biệt, bí ẩn"},
    "M10": {"name": "M10 (Đạo sĩ / Cao nhân)", "seed": 1010, "instruct": "male, elderly, calm, airy", "desc": "Tiên nhân đạo mạo, tĩnh lặng, ung dung"},

    # === NHÂN VẬT NỮ (F1 - F10) ===
    "F1": {"name": "F1 (Nữ chính - Ngọt ngào)", "seed": 2001, "instruct": "female, young adult, high pitch", "desc": "Nữ trẻ ngọt ngào, trong trẻo, dịu dàng"},
    "F2": {"name": "F2 (Nữ phụ / Mẹ / Phu nhân)", "seed": 2002, "instruct": "female, middle-aged, moderate pitch", "desc": "Nữ trung niên điềm đạm, ấm áp"},
    "F3": {"name": "F3 (Bé gái / Trẻ em)", "seed": 2003, "instruct": "female, child, high pitch", "desc": "Bé gái dễ thương, nhí nhảnh"},
    "F4": {"name": "F4 (Nữ bí ẩn / Thì thầm)", "seed": 2004, "instruct": "female, young adult, whisper", "desc": "Nữ thì thầm bí ẩn, nhẹ nhàng"},
    "F5": {"name": "F5 (Bà lão / Lão bà bà)", "seed": 2005, "instruct": "female, elderly, moderate pitch", "desc": "Bà lão già nua, từ tốn"},
    "F6": {"name": "F6 (Tiểu thư / Kiêu kỳ)", "seed": 2006, "instruct": "female, young adult, clear, arrogant", "desc": "Tiểu thư sắc sảo, kiêu kỳ, đài các"},
    "F7": {"name": "F7 (Nữ sát thủ / Lạnh lùng)", "seed": 2007, "instruct": "female, young adult, cold, low pitch", "desc": "Nữ trầm lạnh, dứt khoát, sắc bén"},
    "F8": {"name": "F8 (Nữ hiền dịu / Trầm tính)", "seed": 2008, "instruct": "female, adult, soft, gentle", "desc": "Nữ dịu dàng, trầm tĩnh, sâu lắng"},
    "F9": {"name": "F9 (Hoạt bát / Năng động)", "seed": 2009, "instruct": "female, teenager, lively, cheerful", "desc": "Thiếu nữ năng động, hồn nhiên, vui tươi"},
    "F10": {"name": "F10 (Hoàng hậu / Nữ vương)", "seed": 2010, "instruct": "female, adult, majestic, noble", "desc": "Quý phái, uy nghiêm, quyền lực"},
}

# ==============================================================================
# 🧬 KHO LƯU TRỮ GIỌNG CLONE TRỰC TIẾP TRONG CODE (EMBEDDED CLONE VOICES)
# Dành cho ai muốn lưu VĨNH VIỄN file giọng clone (.pt) thẳng vào Notebook mà không sợ Kaggle xóa!
# Cách lấy mã: Vào Tab 2 hoặc Tab 3 -> Bấm "Xuất Mã Base64 Để Nhúng Vào Code" -> Dán vào đây:
# Ví dụ:
# EMBEDDED_CLONE_VOICES = {
#     "M1": "chuỗi_base64...",
#     "F1": "chuỗi_base64...",
# }
# ==============================================================================
# 🎭 KHO GIỌNG CLONE (ĐÃ NÉN ZLIB TỐI ƯU - GỌN GÀNG, KHÔNG GÂY LAG TRÌNH DUYỆT)
import zlib, json, base64
_VOICES_BLOB = (
    "..."
    "z+u+vt8336jN//f//L//D+gXryk="
)
EMBEDDED_CLONE_VOICES = json.loads(zlib.decompress(base64.b64decode(_VOICES_BLOB)).decode('utf-8'))

# File cấu hình JSON lưu vĩnh viễn Seed và Instruct của từng nhân vật trên đĩa
CHAR_CONFIG_FILE = os.path.join(CACHE_DIR, "character_config.json")

def load_character_config():
    global CHARACTER_DEFAULTS
    if os.path.exists(CHAR_CONFIG_FILE):
        try:
            with open(CHAR_CONFIG_FILE, "r", encoding="utf-8") as f:
                saved = json.load(f)
                for k, v in saved.items():
                    if k in CHARACTER_DEFAULTS:
                        CHARACTER_DEFAULTS[k].update(v)
                    else:
                        CHARACTER_DEFAULTS[k] = v
        except Exception as e:
            print(f"⚠️ Không đọc được cấu hình {CHAR_CONFIG_FILE}: {e}")

def save_character_config():
    try:
        with open(CHAR_CONFIG_FILE, "w", encoding="utf-8") as f:
            json.dump(CHARACTER_DEFAULTS, f, ensure_ascii=False, indent=2)
    except Exception as e:
        print(f"⚠️ Không lưu được cấu hình {CHAR_CONFIG_FILE}: {e}")

# Tự động nạp cấu hình đã lưu trước đó nếu có
load_character_config()
unpack_embedded_clones()

SUPPORTED_LANGS = [
    "Tiếng Việt (vi)", "Tiếng Hàn (ko)", "English (en)", "Tiếng Nhật (ja)", "Tiếng Trung (zh)",
    "Tiếng Tây Ban Nha (es)", "Tiếng Pháp (fr)", "Tiếng Đức (de)", "Tiếng Ý (it)",
    "Tiếng Bồ Đào Nha (pt)", "Tiếng Nga (ru)", "Tiếng Hindi (hi)", "Tiếng Ả Rập (ar)",
    "Tiếng Indonesia (id)", "Tiếng Thái (th)", "Tiếng Hà Lan (nl)"
]

ANCHOR_TEXT_BY_LANG = {
    "vi": "Xin chào, đây là câu thoại mẫu cố định chất giọng chuẩn của nhân vật.",
    "ko": "안녕하세요, 캐릭터 표준 음성 고정용 샘플 대사입니다.",
    "en": "Hello, this is the sample line to establish a consistent voice.",
    "ja": "こんにちは、キャラクターの標準的な声を固定するセリフです。",
    "zh": "你好，这是固定角色标准音色的示例台词。",
    "es": "Hola, esta es la frase de muestra para establecer una voz consistente.",
    "fr": "Bonjour, ceci est la phrase type pour établir une voix cohérente.",
    "de": "Hallo, dies ist der Beispielsatz zur Festlegung einer einheitlichen Stimme.",
    "it": "Ciao, questa è la frase di esempio per stabilire una voce coerente.",
    "pt": "Olá, esta é a frase de exemplo para estabelecer uma voz consistente.",
    "ru": "Здравствуйте, это образцовая фраза для фиксации голоса персонажа.",
    "hi": "नमस्ते, यह चरित्र की आवाज़ को स्थिर करने के लिए नमूना पंक्ति है।",
    "ar": "مرحباً، هذه هي الجملة النموذجية لتثبيت نبرة صوت الشخصية.",
    "id": "Halo, ini adalah kalimat contoh để menetapkan suara karakter yang konsisten.",
    "th": "สวัสดี นี่คือประโยคตัวอย่างเพื่อกำหนดน้ำเสียง củaตัวละครให้คงที่",
    "nl": "Hallo, dit is de voorbeeldzin om een consistente stem vast te stellen."
}

def resolve_lang(lang_choice):
    if not lang_choice or lang_choice == "Auto":
        return None
    lc = str(lang_choice).lower().strip()
    if "việt" in lc or "vie" in lc or "vi" in lc: return "vi"
    if "hàn" in lc or "korean" in lc or "ko" in lc: return "ko"
    if "en" in lc or "anh" in lc or "english" in lc: return "en"
    if "zh" in lc or "trung" in lc or "chinese" in lc: return "zh"
    if "ja" in lc or "nhật" in lc or "japanese" in lc: return "ja"
    if "es" in lc or "tây ban nha" in lc or "spanish" in lc: return "es"
    if "fr" in lc or "pháp" in lc or "french" in lc: return "fr"
    if "de" in lc or "đức" in lc or "german" in lc: return "de"
    if "it" in lc or "ý" in lc or "italian" in lc: return "it"
    if "pt" in lc or "bồ đào nha" in lc or "portuguese" in lc: return "pt"
    if "ru" in lc or "nga" in lc or "russian" in lc: return "ru"
    if "hi" in lc or "hindi" in lc: return "hi"
    if "ar" in lc or "ả rập" in lc or "arabic" in lc: return "ar"
    if "id" in lc or "indonesia" in lc: return "id"
    if "th" in lc or "thái" in lc or "thai" in lc: return "th"
    if "nl" in lc or "hà lan" in lc or "dutch" in lc: return "nl"
    return "vi"

def get_character_pt_path(tag, lang="vi"):
    """Tìm đường dẫn file .pt của nhân vật theo ngôn ngữ, tự động fallback nếu chưa có."""
    tag = tag.upper().strip()
    lang_code = resolve_lang(lang) or "vi"
    lang_pt = os.path.join(CACHE_DIR, lang_code, f"{tag}.pt")
    if os.path.exists(lang_pt):
        return lang_pt
    root_pt = os.path.join(CACHE_DIR, f"{tag}.pt")
    if os.path.exists(root_pt):
        return root_pt
    vi_pt = os.path.join(CACHE_DIR, "vi", f"{tag}.pt")
    if os.path.exists(vi_pt):
        return vi_pt
    return lang_pt

def get_character_save_paths(tag, lang="vi"):
    """Lấy đường dẫn đích để lưu file .pt và .wav cho nhân vật theo ngôn ngữ."""
    tag = tag.upper().strip()
    lang_code = resolve_lang(lang) or "vi"
    lang_dir = os.path.join(CACHE_DIR, lang_code)
    os.makedirs(lang_dir, exist_ok=True)
    dest_pt = os.path.join(lang_dir, f"{tag}.pt")
    dest_wav = os.path.join(lang_dir, f"{tag}_anchor.wav")
    return dest_pt, dest_wav

PROMPT_CACHE = {}

def fix_random_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

def preprocess_reference_audio(audio_path):
    """
    Chuẩn hóa và làm sạch file audio tham chiếu trước khi clone:
    - Kiểm tra file tồn tại, không rỗng
    - Dùng pydub/ffmpeg tự động giải mã mọi định dạng (MP3, M4A, WEBM, AAC, WAV, FLAC, MP4...)
    - Chuyển về Mono 24000Hz WAV chuẩn 100% cho OmniVoice
    - Kiểm tra độ dài: nếu < 0.5s thì cảnh báo, nếu > 20s thì cắt lấy 15s đầu tránh tràn VRAM
    """
    if not audio_path:
        raise ValueError("Chưa tải file âm thanh mẫu lên!")
    if not os.path.exists(audio_path):
        raise ValueError(f"Không tìm thấy file âm thanh: {audio_path}")
    
    file_size = os.path.getsize(audio_path)
    if file_size < 100:
        raise ValueError(f"File âm thanh bị rỗng (0 bytes hoặc chỉ có {file_size} bytes)! Vui lòng kiểm tra lại file của bạn.")

    clean_wav = os.path.abspath("clean_ref_audio.wav")
    try:
        seg = AudioSegment.from_file(audio_path)
        duration_sec = len(seg) / 1000.0
        if duration_sec < 0.5:
            raise ValueError(f"File âm thanh quá ngắn ({duration_sec:.1f}s)! Vui lòng chọn đoạn âm thanh từ 2-10 giây để AI nhận diện được đặc tính giọng.")
        
        # Cắt nếu quá dài (> 20s) để tránh tràn bộ nhớ VRAM GPU
        if duration_sec > 20.0:
            seg = seg[:15000]
            
        seg = seg.set_channels(1).set_frame_rate(24000)
        seg.export(clean_wav, format="wav")
        return clean_wav
    except Exception as e_pydub:
        try:
            data, sr = sf.read(audio_path, dtype="float32")
            if data.size < 2400:
                raise ValueError("File âm thanh mẫu không có dữ liệu sóng âm (0 samples)!")
            sf.write(clean_wav, data, 24000)
            return clean_wav
        except Exception:
            raise ValueError(f"Không thể giải mã file âm thanh ({e_pydub}). Hãy đổi sang định dạng .wav hoặc .mp3 rồi thử lại!")

def get_or_create_character_prompt(tag, custom_instruct="", seed=None, custom_audio=None, lang="vi", force_recreate=False):
    tag = tag.upper().strip()
    lang_code = resolve_lang(lang) or "vi"
    pt_path = get_character_pt_path(tag, lang_code)
    dest_pt, dest_wav = get_character_save_paths(tag, lang_code)
    cache_key = f"{lang_code}_{tag}"

    if not force_recreate and cache_key in PROMPT_CACHE:
        return PROMPT_CACHE[cache_key], dest_wav if os.path.exists(dest_wav) else None

    # 1. Custom audio
    if custom_audio and os.path.exists(custom_audio):
        clean_aud = preprocess_reference_audio(custom_audio)
        try:
            prompt = model.create_voice_clone_prompt(ref_audio=clean_aud, preprocess_prompt=True)
        except Exception:
            prompt = model.create_voice_clone_prompt(ref_audio=clean_aud, preprocess_prompt=False)
        prompt.save(dest_pt)
        shutil.copy(clean_aud, dest_wav)
        PROMPT_CACHE[cache_key] = prompt
        return prompt, dest_wav

    # 2. Đã có file .pt lưu sẵn trên đĩa (theo ngôn ngữ hoặc fallback)
    if not force_recreate and os.path.exists(pt_path):
        try:
            prompt = VoiceClonePrompt.load(pt_path)
            PROMPT_CACHE[cache_key] = prompt
            return prompt, dest_wav if os.path.exists(dest_wav) else (pt_path.replace(".pt", "_anchor.wav") if os.path.exists(pt_path.replace(".pt", "_anchor.wav")) else None)
        except Exception as e:
            print(f"⚠️ Đọc cache {pt_path} lỗi ({e}), sẽ tạo lại...")

    # 3. Tạo mới từ seed + instruct
    defaults = CHARACTER_DEFAULTS.get(tag, {})
    raw_inst = custom_instruct if (custom_instruct and str(custom_instruct).strip()) else defaults.get("instruct", "male, young adult, low pitch" if tag.startswith("M") else "female, young adult, high pitch")
    instruct = sanitize_instruct(raw_inst, default="male, young adult, low pitch" if tag.startswith("M") else "female, young adult, high pitch")
    char_seed = int(seed) if seed is not None else defaults.get("seed", 1000 + (hash(tag) % 5000))

    fix_random_seed(char_seed)
    anchor_text = ANCHOR_TEXT_BY_LANG.get(lang_code, ANCHOR_TEXT_BY_LANG["vi"])
    # 12 bước suy luận cho câu mẫu neo là đủ chuẩn 100% âm sắc mà tốc độ nhanh gấp 3 lần
    anchor_gen_config = OmniVoiceGenerationConfig(num_step=12, guidance_scale=2.0, denoise=True)

    gen_kw = {
        "text": anchor_text,
        "instruct": instruct,
        "generation_config": anchor_gen_config,
    }
    if lang_code:
        gen_kw["language"] = lang_code

    with torch.inference_mode():
        res = model.generate(**gen_kw)
        anchor_clip = res[0]
        sf.write(dest_wav, anchor_clip, model.sampling_rate)
        try:
            prompt = model.create_voice_clone_prompt(ref_audio=dest_wav, ref_text=anchor_text, preprocess_prompt=True)
        except Exception:
            prompt = model.create_voice_clone_prompt(ref_audio=dest_wav, ref_text=anchor_text, preprocess_prompt=False)
        prompt.save(dest_pt)
        PROMPT_CACHE[cache_key] = prompt
    return prompt, dest_wav

SAMPLE_SRT_REVIEW = """1
00:00:00,500 --> 00:00:04,000
M1: Đây là câu chuyện về một chàng trai trẻ quyết tâm thay đổi vận mệnh của mình.

2
00:00:04,500 --> 00:00:08,000
M1: Sư phụ, con nhất định sẽ luyện thành tuyệt kỹ này để bảo vệ môn phái!

3
00:00:08,800 --> 00:00:12,200
M3: Rất tốt, nhưng con tuyệt đối không được kiêu ngạo, giang hồ hiểm ác khôn lường.

4
00:00:13,000 --> 00:00:16,500
F1: Sư huynh, muội tin huynh nhất định sẽ làm được mà!

5
00:00:17,200 --> 00:00:21,000
M2: Hừ, một lũ sâu bọ, các ngươi nghĩ có thể thoát khỏi tay ta sao?

6
00:00:21,800 --> 00:00:25,500
F2: Con trai, hãy cẩn thận, bọn chúng đông người lắm!"""

def parse_time_str(t_str):
    t_str = t_str.strip().replace(',', '.')
    parts = t_str.split(':')
    if len(parts) == 3:
        h, m, s = parts
        return int(h) * 3600 + int(m) * 60 + float(s)
    elif len(parts) == 2:
        m, s = parts
        return int(m) * 60 + float(s)
    return float(parts[0])

def parse_review_srt(content_text):
    segments = []
    blocks = re.split(r'\n\s*\n', content_text.strip())
    is_srt = False
    if len(blocks) > 0 and re.search(r'\d{2}:\d{2}:\d{2}', blocks[0]):
        is_srt = True

    tag_regex = re.compile(
        r'^\s*[\(\[\{]?(M\d+|F\d+|C\d+|MC|DẪN\s*CHUYỆN|DAN\s*CHUYEN|DẪN)[\)\]\}]?\s*[:\|\-\s]\s*(.*)$',
        re.DOTALL | re.IGNORECASE
    )

    if is_srt:
        for idx, block in enumerate(blocks):
            lines = [l.strip() for l in block.splitlines() if l.strip()]
            if len(lines) >= 2:
                time_line_idx = -1
                for i, l in enumerate(lines):
                    if '-->' in l:
                        time_line_idx = i
                        break
                if time_line_idx != -1:
                    times = lines[time_line_idx].split('-->')
                    start_s = parse_time_str(times[0])
                    end_s = parse_time_str(times[1])
                    text_content = ' '.join(lines[time_line_idx+1:])
                    m = tag_regex.match(text_content)
                    if m:
                        raw_tag = m.group(1).upper()
                        tag = 'M1' if ('DẪN' in raw_tag or 'DAN' in raw_tag or raw_tag == 'MC') else raw_tag
                        dialogue = m.group(2).strip()
                    else:
                        tag = 'M1'
                        dialogue = text_content.strip()
                    segments.append({
                        'index': idx + 1, 'start': start_s, 'end': end_s,
                        'tag': tag, 'text': dialogue
                    })
    else:
        lines = [l.strip() for l in content_text.splitlines() if l.strip()]
        for idx, line in enumerate(lines):
            m = tag_regex.match(line)
            if m:
                raw_tag = m.group(1).upper()
                tag = 'M1' if ('DẪN' in raw_tag or 'DAN' in raw_tag or raw_tag == 'MC') else raw_tag
                dialogue = m.group(2).strip()
            else:
                tag = 'M1'
                dialogue = line
            segments.append({
                'index': idx + 1, 'start': None, 'end': None,
                'tag': tag, 'text': dialogue
            })

    return segments, is_srt

def generate_review_dubbing(
    raw_script, uploaded_file, lang_choice, lock_voice_enabled, align_timeline, silence_gap, speed, step_choice,
    m1_voice, m2_voice, m3_voice, m4_voice, m5_voice, m6_voice, m7_voice, m8_voice, m9_voice, m10_voice,
    f1_voice, f2_voice, f3_voice, f4_voice, f5_voice, f6_voice, f7_voice, f8_voice, f9_voice, f10_voice,
    m1_seed, m2_seed, m3_seed, m4_seed, m5_seed, m6_seed, m7_seed, m8_seed, m9_seed, m10_seed,
    f1_seed, f2_seed, f3_seed, f4_seed, f5_seed, f6_seed, f7_seed, f8_seed, f9_seed, f10_seed,
    m1_clone_audio, f1_clone_audio,
    progress=gr.Progress()
):
    if uploaded_file is not None:
        with open(uploaded_file.name, "r", encoding="utf-8", errors="ignore") as f:
            content = f.read()
    else:
        content = raw_script

    if not content or not content.strip():
        return None, "⚠️ Vui lòng nhập kịch bản hoặc tải lên file SRT!", None

    segments, is_srt = parse_review_srt(content)
    if not segments:
        return None, "⚠️ Không tìm thấy câu thoại hợp lệ nào!", None

    cfg_list = [m1_voice, m2_voice, m3_voice, m4_voice, m5_voice, m6_voice, m7_voice, m8_voice, m9_voice, m10_voice,
                f1_voice, f2_voice, f3_voice, f4_voice, f5_voice, f6_voice, f7_voice, f8_voice, f9_voice, f10_voice]
    seed_list = [m1_seed, m2_seed, m3_seed, m4_seed, m5_seed, m6_seed, m7_seed, m8_seed, m9_seed, m10_seed,
                 f1_seed, f2_seed, f3_seed, f4_seed, f5_seed, f6_seed, f7_seed, f8_seed, f9_seed, f10_seed]
    all_tags = [f"M{i}" for i in range(1, 11)] + [f"F{i}" for i in range(1, 11)]

    voice_map = {}
    seed_map = {}
    for tag, cfg_val, s_val in zip(all_tags, cfg_list, seed_list):
        def_inst = CHARACTER_DEFAULTS.get(tag, {}).get("instruct", "male, young adult, low pitch" if tag.startswith("M") else "female, young adult, high pitch")
        voice_map[tag] = sanitize_instruct(cfg_val, def_inst)
        seed_map[tag] = s_val

    clone_map = {"M1": m1_clone_audio, "F1": f1_clone_audio}

    sampling_rate = model.sampling_rate
    audio_clips = []
    log_lines = []
    total = len(segments)

    lang_code = resolve_lang(lang_choice)
    num_step = 16
    if "10" in str(step_choice):
        num_step = 10
    elif "32" in str(step_choice):
        num_step = 32

    gen_config = OmniVoiceGenerationConfig(num_step=num_step, guidance_scale=2.0, denoise=True)

    unique_tags = list(set(s['tag'] for s in segments))
    tag_prompts = {}
    total_ops = len(unique_tags) + total + 1

    if lock_voice_enabled:
        for idx, tag in enumerate(unique_tags):
            pt_path = os.path.join(CACHE_DIR, f"{tag}.pt")
            custom_inst = voice_map.get(tag, "")
            char_seed = seed_map.get(tag, None)
            ref_aud = clone_map.get(tag, None)

            saved_seed = CHARACTER_DEFAULTS.get(tag, {}).get("seed")
            seed_changed = (char_seed is not None and saved_seed is not None and int(char_seed) != int(saved_seed))
            has_clone = os.path.exists(pt_path)
            # Bảo vệ tuyệt đối file giọng đã có: KHÔNG tự ý ghi đè khi lồng tiếng SRT trừ khi có audio mới
            should_recreate = False
            if ref_aud and os.path.exists(ref_aud):
                should_recreate = True
            elif seed_changed and not has_clone:
                should_recreate = True

            desc_status = "nạp tức thì từ cache" if has_clone else ("tạo mới theo seed" if should_recreate else "tạo khuôn giọng")
            progress(idx / total_ops, desc=f"🔒 [{idx+1}/{len(unique_tags)}] Khóa giọng [{tag}] ({desc_status})...")

            prompt_obj, _ = get_or_create_character_prompt(
                tag=tag, custom_instruct=custom_inst, seed=char_seed, custom_audio=ref_aud,
                lang=lang_code if lang_code else "vi", force_recreate=should_recreate
            )
            tag_prompts[tag] = prompt_obj
            if char_seed is not None and not has_clone:
                if tag not in CHARACTER_DEFAULTS: CHARACTER_DEFAULTS[tag] = {}
                CHARACTER_DEFAULTS[tag]["seed"] = int(char_seed)
        save_character_config()

    for i, seg in enumerate(segments):
        prog = (len(unique_tags) + i) / total_ops
        tag = seg['tag']
        text = seg['text']
        short_txt = text[:25] + "..." if len(text) > 25 else text
        progress(prog, desc=f"🎙️ [{i+1}/{total}] Lồng tiếng [{tag}]: '{short_txt}'")
        
        kw = {"text": text, "generation_config": gen_config}
        if lang_code:
            kw["language"] = lang_code
        if speed != 1.0:
            kw["speed"] = float(speed)

        if lock_voice_enabled and tag in tag_prompts:
            kw["voice_clone_prompt"] = tag_prompts[tag]
        else:
            ref_aud = clone_map.get(tag)
            if ref_aud:
                with torch.inference_mode():
                    kw["voice_clone_prompt"] = model.create_voice_clone_prompt(ref_audio=ref_aud)
            else:
                custom_inst = voice_map.get(tag, "")
                kw["instruct"] = sanitize_instruct(custom_inst, "male, young adult, low pitch" if tag.startswith("M") else "female, young adult, high pitch")

        try:
            with torch.inference_mode():
                res = model.generate(**kw)
            clip = res[0]
            dur = len(clip) / sampling_rate
            audio_clips.append({"seg": seg, "audio": clip, "duration": dur})
            mode_desc = "🔒 Đã khóa giọng" if (lock_voice_enabled and tag in tag_prompts) else "🎲 Giọng tự do"
            log_lines.append(f"[{tag}] #{seg['index']} ({mode_desc}): '{text[:28]}...' ({dur:.2f}s)")
        except Exception as e:
            log_lines.append(f"❌ Lỗi câu #{seg['index']} [{tag}]: {e}")

    if not audio_clips:
        return None, "Lỗi khi sinh âm thanh!", None

    progress(0.95, desc="Đang ghép nối toàn bộ âm thanh kịch bản...")
    if is_srt and align_timeline:
        full_track = []
        current_time = 0.0
        for item in audio_clips:
            seg = item["seg"]
            target_start = seg["start"]
            if target_start is not None and target_start > current_time:
                silence_dur = target_start - current_time
                silence_samples = int(silence_dur * sampling_rate)
                full_track.append(np.zeros(silence_samples, dtype=np.float32))
                current_time = target_start
            full_track.append(item["audio"])
            current_time += item["duration"]
        final_waveform = np.concatenate(full_track)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    else:
        full_track = []
        pause_samples = int(silence_gap * sampling_rate)
        pause_wav = np.zeros(pause_samples, dtype=np.float32)
        for idx, item in enumerate(audio_clips):
            full_track.append(item["audio"])
            if idx < len(audio_clips) - 1:
                full_track.append(pause_wav)
        final_waveform = np.concatenate(full_track)

    # Lưu trực tiếp vào thư mục hiện hành để Gradio cấp quyền truy cập an toàn 100%
    out_file = os.path.abspath("omnivoice_review_output.wav")
    sf.write(out_file, final_waveform, sampling_rate)
    try:
        backup_file = os.path.join(BASE_DIR, "omnivoice_review_output.wav")
        if os.path.abspath(backup_file) != out_file:
            shutil.copy(out_file, backup_file)
    except Exception:
        pass
    waveform_int16 = (final_waveform * 32767).clip(-32768, 32767).astype(np.int16)

    status_report = f"✅ Hoàn tất lồng tiếng {len(audio_clips)}/{total} câu thoại! (Khóa Giọng: {'BẬT 🔒' if lock_voice_enabled else 'TẮT 🎲'})\n" + "\n".join(log_lines[:30])
    if len(log_lines) > 30:
        status_report += f"\n... và {len(log_lines)-30} câu thoại khác."

    return (sampling_rate, waveform_int16), status_report, out_file

# Các hàm xử lý trong Tab Quản Lý & Khóa Giọng Chuẩn
def get_single_char_pt(tag, lang="vi"):
    tag = tag.upper().strip()
    pt_path = get_character_pt_path(tag, lang)
    local_pt = os.path.abspath(f"{tag}.pt")
    if os.path.exists(pt_path):
        try:
            if pt_path != local_pt:
                shutil.copy(pt_path, local_pt)
            return local_pt
        except Exception:
            return pt_path
    return None

def get_single_char_wav(tag, lang="vi"):
    tag = tag.upper().strip()
    lang_code = resolve_lang(lang) or "vi"
    wav_path = os.path.join(CACHE_DIR, lang_code, f"{tag}_anchor.wav")
    if not os.path.exists(wav_path):
        wav_path = os.path.join(CACHE_DIR, f"{tag}_anchor.wav")
    local_wav = os.path.abspath(f"{tag}_giong_chuan.wav")
    if os.path.exists(wav_path):
        try:
            if wav_path != local_wav:
                shutil.copy(wav_path, local_wav)
            return local_wav
        except Exception:
            return wav_path
    return None

def on_select_character(tag, lang="vi"):
    load_character_config()
    tag = tag.upper().strip()
    lang_code = resolve_lang(lang) or "vi"
    defaults = CHARACTER_DEFAULTS.get(tag, {})
    seed = defaults.get("seed", 1001)
    inst = defaults.get("instruct", "")
    desc = defaults.get("desc", "")
    
    dest_pt, dest_wav = get_character_save_paths(tag, lang_code)
    pt_path = get_character_pt_path(tag, lang_code)
    wav_path = dest_wav if os.path.exists(dest_wav) else os.path.join(CACHE_DIR, f"{tag}_anchor.wav")
    audio_val = wav_path if os.path.exists(wav_path) else None
    info = f"**{tag}** ({lang_code.upper()}): {desc}"

    pt_file = get_single_char_pt(tag, lang_code)
    wav_file = get_single_char_wav(tag, lang_code)

    is_embedded = ("EMBEDDED_CLONE_VOICES" in globals() and isinstance(EMBEDDED_CLONE_VOICES, dict) and tag in EMBEDDED_CLONE_VOICES)
    has_pt = os.path.exists(dest_pt)
    has_fallback = os.path.exists(pt_path)

    if has_pt:
        status_msg = f"🛡️ [GIỌNG CLONE ĐỘC LẬP CHO {lang_code.upper()}]\nNhân vật [{tag}] đã có file giọng riêng cho {lang_code.upper()} ({dest_pt}).\n👉 Bấm '🎧 Nghe Thử Giọng Hiện Tại' để nghe chất giọng chuẩn của [{tag}]!"
    elif is_embedded or has_fallback:
        status_msg = f"🌐 [DÙNG GIỌNG GỐC CHUẨN (FALLBACK)]\nNhân vật [{tag}] đang dùng giọng gốc của hệ thống. Bạn có thể tải file audio để lưu giọng riêng cho quốc gia {lang_code.upper()}!"
    else:
        status_msg = f"ℹ️ Nhân vật [{tag}] chưa có file giọng chuẩn. Bạn có thể nghe thử với Seed bên dưới và bấm 'Lưu Giọng Này'."

    return seed, inst, info, audio_val, status_msg, pt_file, wav_file

def play_character_current_voice(tag, preview_text, lang_choice, progress=gr.Progress()):
    tag = tag.upper().strip()
    lang_code = resolve_lang(lang_choice) or "vi"
    pt_path = get_character_pt_path(tag, lang_code)
    dest_pt, dest_wav = get_character_save_paths(tag, lang_code)
    cache_key = f"{lang_code}_{tag}"
    
    progress(0.2, desc=f"Đang chuẩn bị giọng [{tag}] ({lang_code.upper()})...")
    try:
        # Nếu chưa có file .pt, tạo lần đầu từ seed + instruct
        if not os.path.exists(pt_path):
            prompt_obj, wav_p = get_or_create_character_prompt(tag=tag, lang=lang_code, force_recreate=False)
        else:
            if cache_key in PROMPT_CACHE:
                prompt_obj = PROMPT_CACHE[cache_key]
            else:
                prompt_obj = VoiceClonePrompt.load(pt_path)
                PROMPT_CACHE[cache_key] = prompt_obj

        # Đọc câu thoại mẫu bằng chính file giọng chuẩn đang lưu (Bảo toàn 100%, KHÔNG đổi giọng)
        progress(0.6, desc=f"Đang phát câu thoại bằng giọng [{tag}]...")
        defaults = CHARACTER_DEFAULTS.get(tag, {})
        char_name = defaults.get("name", tag)
        gen_text = preview_text.strip() if (preview_text and preview_text.strip()) else ANCHOR_TEXT_BY_LANG.get(lang_code, f"Xin chào, tôi là {char_name}, đây là câu thoại thử nghiệm chất giọng chuẩn của tôi.")
        gen_cfg = OmniVoiceGenerationConfig(num_step=16, guidance_scale=2.0, denoise=True)
        gen_kw = {
            "text": gen_text,
            "voice_clone_prompt": prompt_obj,
            "generation_config": gen_cfg,
        }
        if lang_code:
            gen_kw["language"] = lang_code

        with torch.inference_mode():
            res = model.generate(**gen_kw)
        clip = res[0]
        sf.write(dest_wav, clip, model.sampling_rate)

        pt_file = get_single_char_pt(tag, lang_code)
        wav_file = get_single_char_wav(tag, lang_code)
        msg = f"🎧 Đang phát giọng [{tag}] kho ({lang_code.upper()})!\n🛡️ File giọng ({os.path.basename(pt_path)}) ĐƯỢC BẢO TOÀN NGUYÊN VẸN 100%."
        return dest_wav, msg, pt_file, wav_file
    except Exception as e:
        return None, f"❌ Lỗi khi phát giọng [{tag}]: {e}", None, None

def preview_temporary_voice(tag, seed, instruct, preview_text, audio_file, lang_choice, progress=gr.Progress()):
    tag = tag.upper().strip()
    lang_code = resolve_lang(lang_choice) or "vi"
    progress(0.3, desc=f"Đang tạo bản nghe thử tạm thời cho [{tag}] ({lang_code.upper()})...")
    try:
        clean_inst = sanitize_instruct(instruct, default="male, young adult, low pitch" if tag.startswith("M") else "female, young adult, high pitch")
        char_seed = int(seed) if seed is not None else 1001

        # Nếu có tải file audio người thật lên để nghe thử clone
        if audio_file and os.path.exists(audio_file):
            clean_aud = preprocess_reference_audio(audio_file)
            try:
                temp_prompt = model.create_voice_clone_prompt(ref_audio=clean_aud, preprocess_prompt=True)
            except Exception:
                temp_prompt = model.create_voice_clone_prompt(ref_audio=clean_aud, preprocess_prompt=False)
            gen_text = preview_text.strip() if (preview_text and preview_text.strip()) else ANCHOR_TEXT_BY_LANG.get(lang_code, "Xin chào, đây là bản nghe thử nhái theo file âm thanh vừa tải lên.")
            gen_cfg = OmniVoiceGenerationConfig(num_step=16, guidance_scale=2.0, denoise=True)
            gen_kw = {"text": gen_text, "voice_clone_prompt": temp_prompt, "generation_config": gen_cfg}
            if lang_code: gen_kw["language"] = lang_code
            with torch.inference_mode():
                res = model.generate(**gen_kw)
            temp_wav = os.path.join(CACHE_DIR, f"temp_test_{tag}.wav")
            sf.write(temp_wav, res[0], model.sampling_rate)
            msg = f"🎧 Đang nghe thử giọng clone từ file tải lên cho {lang_code.upper()}.\n🛡️ Giọng chính thức của [{tag}] VẪN ĐƯỢC GIỮ NGUYÊN (Chưa lưu). Bấm 'Lưu Giọng Mới Này' nếu bạn ưng ý!"
            return temp_wav, msg

        # Thử nghiệm với Seed và Instruct (CHỈ TẠO FILE TẠM - TUYỆT ĐỐI KHÔNG GHI ĐÈ FILE GỐC)
        fix_random_seed(char_seed)
        gen_text = preview_text.strip() if (preview_text and preview_text.strip()) else ANCHOR_TEXT_BY_LANG.get(lang_code, f"Xin chào, đây là câu thoại nghe thử với Seed {char_seed}.")
        anchor_gen_config = OmniVoiceGenerationConfig(num_step=12, guidance_scale=2.0, denoise=True)
        gen_kw = {
            "text": gen_text,
            "instruct": clean_inst,
            "generation_config": anchor_gen_config,
        }
        if lang_code: gen_kw["language"] = lang_code
        with torch.inference_mode():
            res = model.generate(**gen_kw)
        temp_wav = os.path.join(CACHE_DIR, f"temp_test_{tag}.wav")
        sf.write(temp_wav, res[0], model.sampling_rate)

        msg = f"🎧 Đang nghe thử với Seed {char_seed} ({lang_code.upper()}).\n🛡️ File giọng chính thức của [{tag}] VẪN ĐƯỢC BẢO VỆ NGUYÊN VẸN (Chưa lưu).\n👉 Nếu bạn ưng ý, hãy bấm '💾 Áp Dụng & Lưu Làm Giọng Chuẩn Mới'."
        return temp_wav, msg
    except Exception as e:
        return None, f"❌ Lỗi khi nghe thử: {e}"

def confirm_save_character_voice(tag, seed, instruct, audio_file, lang_choice, progress=gr.Progress()):
    tag = tag.upper().strip()
    lang_code = resolve_lang(lang_choice) or "vi"
    progress(0.4, desc=f"Đang lưu và khóa giọng mới cho [{tag}] ({lang_code.upper()})...")
    try:
        clean_inst = sanitize_instruct(instruct, default="male, young adult, low pitch" if tag.startswith("M") else "female, young adult, high pitch")
        char_seed = int(seed) if seed is not None else 1001

        # Cập nhật và lưu config
        if tag not in CHARACTER_DEFAULTS:
            CHARACTER_DEFAULTS[tag] = {}
        CHARACTER_DEFAULTS[tag]["seed"] = char_seed
        CHARACTER_DEFAULTS[tag]["instruct"] = clean_inst
        save_character_config()

        prompt, wav_path = get_or_create_character_prompt(
            tag=tag, custom_instruct=clean_inst, seed=char_seed, custom_audio=audio_file,
            lang=lang_code, force_recreate=True
        )
        dest_pt, dest_wav = get_character_save_paths(tag, lang_code)
        pt_file = get_single_char_pt(tag, lang_code)
        wav_file = get_single_char_wav(tag, lang_code)
        msg = f"✅ ĐÃ XÁC NHẬN LƯU THÀNH CÔNG GIỌNG MỚI CHO [{tag}] ({lang_code.upper()})!\nSeed: {char_seed} | File: {dest_pt}\n👉 Khi lồng tiếng kịch bản {lang_code.upper()}, [{tag}] sẽ tự động nói giọng mới này."
        return wav_path, msg, pt_file, wav_file
    except Exception as e:
        return None, f"❌ Lỗi khi lưu giọng cho [{tag}]: {e}", None, None

def restore_original_clone_voice(tag, preview_text, lang_choice, progress=gr.Progress()):
    tag = tag.upper().strip()
    lang_code = resolve_lang(lang_choice) or "vi"
    progress(0.3, desc=f"Đang khôi phục giọng clone gốc cho [{tag}]...")
    if "EMBEDDED_CLONE_VOICES" in globals() and isinstance(EMBEDDED_CLONE_VOICES, dict) and tag in EMBEDDED_CLONE_VOICES:
        b64_str = EMBEDDED_CLONE_VOICES[tag]
        if b64_str and len(b64_str.strip()) > 10:
            dest_pt, dest_wav = get_character_save_paths(tag, lang_code)
            try:
                with open(dest_pt, "wb") as f:
                    f.write(base64.b64decode(b64_str.strip()))
                PROMPT_CACHE[f"{lang_code}_{tag}"] = VoiceClonePrompt.load(dest_pt)

                orig_seed = 15161 if tag == "M1" else (1000 + int(tag[1:])) if tag.startswith("M") else (2000 + int(tag[1:]))
                if tag in CHARACTER_DEFAULTS:
                    CHARACTER_DEFAULTS[tag]["seed"] = orig_seed
                save_character_config()

                # Tự động phát lại giọng gốc để người dùng nghe kiểm chứng ngay
                wav_p, _, pt_f, wav_f = play_character_current_voice(tag, preview_text, lang_choice)
                msg = f"🎉 ĐÃ KHÔI PHỤC THÀNH CÔNG GIỌNG CLONE GỐC CỦA [{tag}] TỪ CODE!\nFile {dest_pt} đã được phục hồi nguyên vẹn 100% (Seed: {orig_seed})."
                return orig_seed, wav_p, msg, pt_f, wav_f
            except Exception as e:
                return None, None, f"❌ Lỗi khi khôi phục giọng [{tag}]: {e}", None, None
    return None, None, f"⚠️ Nhân vật [{tag}] không có trong kho giọng nhúng EMBEDDED_CLONE_VOICES!", None, None

def import_single_char_file(tag, uploaded_file, lang_choice="Tiếng Việt (vi)"):
    if uploaded_file is None:
        return None, "⚠️ Vui lòng chọn file .pt hoặc file audio để nạp!", None, None
    tag = tag.upper().strip()
    lang_code = resolve_lang(lang_choice) or "vi"
    fname = uploaded_file.name.lower()
    dest_pt, dest_wav = get_character_save_paths(tag, lang_code)

    try:
        if fname.endswith(".pt"):
            shutil.copy(uploaded_file.name, dest_pt)
            PROMPT_CACHE[f"{lang_code}_{tag}"] = VoiceClonePrompt.load(dest_pt)
            msg = f"✅ Đã nạp thành công file giọng chuẩn [{tag}.pt] cho {lang_code.upper()}!"
            return dest_pt, msg, dest_pt, None
        else:
            clean_aud = preprocess_reference_audio(uploaded_file.name)
            prompt = model.create_voice_clone_prompt(ref_audio=clean_aud)
            prompt.save(dest_pt)
            shutil.copy(clean_aud, dest_wav)
            PROMPT_CACHE[f"{lang_code}_{tag}"] = prompt
            msg = f"✅ Đã trích xuất và tạo thành công khuôn giọng chuẩn [{tag}.pt] ({lang_code.upper()}) từ file audio!"
            return dest_wav, msg, dest_pt, dest_wav
    except Exception as e:
        return None, f"❌ Lỗi khi nạp file: {e}", None, None

def export_voice_bank_zip():
    save_character_config()
    zip_path = os.path.abspath("omnivoice_character_bank.zip")
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files in os.walk(CACHE_DIR):
            for file in files:
                full_path = os.path.join(root, file)
                rel_path = os.path.relpath(full_path, CACHE_DIR)
                zf.write(full_path, rel_path)
    try:
        backup_zip = os.path.join(BASE_DIR, "omnivoice_character_bank.zip")
        if os.path.abspath(backup_zip) != zip_path:
            shutil.copy(zip_path, backup_zip)
    except Exception:
        pass
    return zip_path

def import_voice_bank_zip(uploaded_zip):
    if uploaded_zip is None:
        return "⚠️ Vui lòng chọn file zip để nạp!"
    try:
        with zipfile.ZipFile(uploaded_zip.name, 'r') as zf:
            zf.extractall(CACHE_DIR)
        PROMPT_CACHE.clear()
        load_character_config()
        files = os.listdir(CACHE_DIR)
        return f"✅ Đã nạp thành công bộ giọng! Tìm thấy {len(files)} files trong {CACHE_DIR}."
    except Exception as e:
        return f"❌ Lỗi khi giải nén: {e}"

# Xử lý Tab 3: Single Voice Clone & Xuất file giọng mẫu .pt
LAST_CLONED_PROMPT = None
LAST_CLONED_AUDIO = None

def process_single_clone(text, ref_audio, ref_text, speed, lang_choice, progress=gr.Progress()):
    global LAST_CLONED_PROMPT, LAST_CLONED_AUDIO
    if not text or not text.strip():
        return None, "⚠️ Vui lòng nhập văn bản cần đọc thử!", None, None
    if ref_audio is None:
        return None, "⚠️ Vui lòng tải lên file âm thanh mẫu người thật (3 - 10 giây)!", None, None
    try:
        progress(0.1, desc="Đang kiểm tra và chuẩn hóa file âm thanh mẫu...")
        clean_audio = preprocess_reference_audio(ref_audio)

        progress(0.3, desc="Đang trích xuất đặc trưng âm sắc (Voice Clone Prompt)...")
        kw_clone = {"ref_audio": clean_audio}
        if ref_text and ref_text.strip():
            kw_clone["ref_text"] = ref_text.strip()
            
        try:
            prompt = model.create_voice_clone_prompt(**kw_clone, preprocess_prompt=True)
        except Exception as e_prep:
            print(f"⚠️ Trích xuất với lọc khoảng lặng thất bại ({e_prep}), tự động chuyển sang chế độ bảo toàn âm thanh...")
            prompt = model.create_voice_clone_prompt(**kw_clone, preprocess_prompt=False)
            
        LAST_CLONED_PROMPT = prompt
        LAST_CLONED_AUDIO = clean_audio

        progress(0.6, desc="Đang tổng hợp giọng nói theo mẫu...")
        gen_cfg = OmniVoiceGenerationConfig(num_step=16, guidance_scale=2.0, denoise=True)
        lang_c = resolve_lang(lang_choice)
        gen_kw = {
            "text": text.strip(),
            "voice_clone_prompt": prompt,
            "generation_config": gen_cfg,
        }
        if lang_c:
            gen_kw["language"] = lang_c
        if speed != 1.0:
            gen_kw["speed"] = float(speed)

        with torch.inference_mode():
            res = model.generate(**gen_kw)
        clip = res[0]
        sr = model.sampling_rate

        # Lưu file .pt để người dùng có thể tải về làm file voice mẫu vĩnh viễn (như F2.pt, F3.pt)
        save_pt_path = os.path.abspath("giong_clone_mau.pt")
        prompt.save(save_pt_path)

        # Lưu file .wav kết quả
        save_wav_path = os.path.abspath("giong_clone_ketqua.wav")
        sf.write(save_wav_path, clip, sr)

        progress(1.0, desc="Hoàn tất!")
        status_msg = "✅ Đã sinh âm thanh nhái giọng thành công!\n👉 Đã tạo sẵn file giọng mẫu (.pt) bên dưới, bạn có thể TẢI VỀ MÁY hoặc GÁN THẲNG cho nhân vật (M1, F1...) để lồng tiếng phim!"
        return save_wav_path, status_msg, save_pt_path, save_wav_path
    except Exception as e:
        traceback.print_exc()
        err_type = type(e).__name__
        err_msg = str(e).strip() if str(e).strip() else "Không thể trích xuất đặc trưng từ file âm thanh này."
        return None, f"❌ Lỗi khi clone ({err_type}): {err_msg}", None, None

def assign_cloned_voice_to_character(target_tag, target_lang="Tiếng Việt (vi)"):
    global LAST_CLONED_PROMPT, LAST_CLONED_AUDIO
    if LAST_CLONED_PROMPT is None:
        return "⚠️ Chưa có giọng nhái nào được tạo! Vui lòng tải file audio mẫu và bấm 'Sinh Giọng Nhái' trước."
    target_tag = target_tag.upper().strip()
    lang_code = resolve_lang(target_lang) or "vi"
    dest_pt, dest_wav = get_character_save_paths(target_tag, lang_code)
    try:
        LAST_CLONED_PROMPT.save(dest_pt)
        PROMPT_CACHE[f"{lang_code}_{target_tag}"] = LAST_CLONED_PROMPT
        if LAST_CLONED_AUDIO and os.path.exists(LAST_CLONED_AUDIO):
            shutil.copy(LAST_CLONED_AUDIO, dest_wav)
        elif os.path.exists(os.path.abspath("giong_clone_ketqua.wav")):
            shutil.copy(os.path.abspath("giong_clone_ketqua.wav"), dest_wav)
        save_character_config()
        return f"🎉 ĐÃ GÁN THÀNH CÔNG CHO [{target_tag}] ({lang_code.upper()})!\n\n✅ Giọng nhái này đã trở thành giọng chuẩn chính thức của [{target_tag}] trong kho {lang_code.upper()} (lưu tại {dest_pt}).\n👉 Khi bạn lồng tiếng phim {lang_code.upper()}, [{target_tag}] sẽ tự động nói đúng giọng này 100%!"
    except Exception as e:
        return f"❌ Lỗi khi gán giọng cho [{target_tag}]: {e}"

def sync_tab1_seeds():
    load_character_config()
    all_tags = [f"M{i}" for i in range(1, 11)] + [f"F{i}" for i in range(1, 11)]
    seeds = [CHARACTER_DEFAULTS.get(t, {}).get("seed", 1000 + (hash(t) % 5000)) for t in all_tags]
    cfgs = [CHARACTER_DEFAULTS.get(t, {}).get("instruct", "male, young adult, low pitch" if t.startswith("M") else "female, young adult, high pitch") for t in all_tags]
    return tuple(seeds + cfgs)

def export_char_base64_code(tag):
    tag = tag.upper().strip()
    pt_path = os.path.join(CACHE_DIR, f"{tag}.pt")
    if not os.path.exists(pt_path):
        return f"⚠️ Nhân vật [{tag}] chưa có file giọng clone (.pt). Vui lòng nạp file audio hoặc tạo giọng trước!"
    try:
        with open(pt_path, "rb") as f:
            b64 = base64.b64encode(f.read()).decode('utf-8')
        code = f"# Dán dòng này vào biến EMBEDDED_CLONE_VOICES ở đầu ô [BƯỚC 2/2] để lưu vĩnh viễn vào Code Notebook:\n"
        code += f'EMBEDDED_CLONE_VOICES["{tag}"] = "{b64}"'
        return code
    except Exception as e:
        return f"❌ Lỗi khi trích xuất Base64: {e}"

def export_last_cloned_base64_code(target_tag):
    global LAST_CLONED_PROMPT
    target_tag = target_tag.upper().strip()
    temp_pt = os.path.abspath(f"temp_export_{target_tag}.pt")
    dest_pt = os.path.join(CACHE_DIR, f"{target_tag}.pt")
    try:
        if LAST_CLONED_PROMPT is not None:
            LAST_CLONED_PROMPT.save(temp_pt)
            with open(temp_pt, "rb") as f:
                b64 = base64.b64encode(f.read()).decode('utf-8')
            if os.path.exists(temp_pt):
                os.remove(temp_pt)
        elif os.path.exists(dest_pt):
            with open(dest_pt, "rb") as f:
                b64 = base64.b64encode(f.read()).decode('utf-8')
        else:
            return "⚠️ Chưa có giọng nhái nào được tạo! Vui lòng tải file audio mẫu và bấm 'Sinh Giọng Nhái' trước."
        code = f"# Dán dòng này vào biến EMBEDDED_CLONE_VOICES ở đầu ô [BƯỚC 2/2] để lưu vĩnh viễn vào Code Notebook:\n"
        code += f'EMBEDDED_CLONE_VOICES["{target_tag}"] = "{b64}"'
        return code
    except Exception as e:
        return f"❌ Lỗi khi xuất Base64: {e}"

def export_character_defaults_code():
    save_character_config()
    res = "# Dán đè đoạn này vào biến CHARACTER_DEFAULTS ở đầu ô [BƯỚC 2/2] để lưu vĩnh viễn vào Code:\n"
    res += "CHARACTER_DEFAULTS = " + json.dumps(CHARACTER_DEFAULTS, ensure_ascii=False, indent=4)
    return res

# KHỞI TẠO GIAO DIỆN WEB BLOCKS CHUẨN GRADIO (KHÔNG LỒNG BLOCKS)
with gr.Blocks(title="OmniVoice Review Phim Studio") as custom_app:
    gr.Markdown(f"""
# 🎬 OmniVoice — Studio Lồng Tiếng Review Phim Chuẩn SRT
### 🔒 Hỗ trợ 20 mã nhân vật: `M1-M10` (Male), `F1-F10` (Female) — Khóa Giọng Chuẩn Vĩnh Viễn Không Bị Random
{gpu_info_banner}
""")

    with gr.Tabs():
        # TAB 1: LỒNG TIẾNG SRT CHÍNH
        with gr.TabItem("🎬 Lồng Tiếng SRT Review Phim (M1-M10, F1-F10)"):
            with gr.Row():
                with gr.Column(scale=1):
                    script_input = gr.Textbox(
                        label="Dán Kịch Bản / Nội Dung SRT vào đây",
                        lines=10,
                        placeholder="1\n00:00:01,000 --> 00:00:04,000\nMC: Đây là câu chuyện...\n\n2\n00:00:04,500 --> 00:00:07,000\nM1: Tôi là nam chính!",
                        value=SAMPLE_SRT_REVIEW
                    )
                    with gr.Row():
                        btn_load_sample = gr.Button("📋 Nạp Kịch Bản Mẫu Review Phim", variant="secondary", size="sm")
                        file_upload = gr.File(label="Hoặc Tải File .SRT lên", file_types=[".srt", ".txt"])

                    btn_load_sample.click(lambda: SAMPLE_SRT_REVIEW, outputs=[script_input])

                    with gr.Accordion("⚙️ Thiết Lập Khóa Giọng, Thời Gian & Tốc Độ", open=True):
                        lock_voice_box = gr.Checkbox(
                            label="🔒 Khóa giọng chuẩn cố định (Mỗi nhân vật giữ nguyên 1 giọng duy nhất xuyên suốt phim)",
                            value=True
                        )
                        align_box = gr.Checkbox(label="Khớp chuẩn mốc thời gian SRT (Chèn khoảng lặng đúng timestamp)", value=True)
                        step_choice = gr.Radio(
                            choices=["⚡ Siêu nhanh (10 bước)", "🎯 Cân bằng - Khuyên dùng (16 bước)", "💎 Kỹ tính (32 bước)"],
                            value="🎯 Cân bằng - Khuyên dùng (16 bước)",
                            label="⚡ Tốc độ suy luận AI (Inference Steps - Càng thấp tạo càng nhanh)"
                        )
                        gap_slider = gr.Slider(0.1, 2.0, value=0.4, step=0.1, label="Khoảng nghỉ giữa các câu (giây) - nếu không dùng SRT")
                        speed_slider = gr.Slider(0.6, 1.5, value=1.0, step=0.05, label="Tốc độ đọc (Speed)")
                        lang_dd = gr.Dropdown(
                            label="Ngôn ngữ kịch bản SRT",
                            choices=SUPPORTED_LANGS + ["Auto"],
                            value="Tiếng Việt (vi)"
                        )

                    btn_generate = gr.Button("🚀 BẮT ĐẦU LỒNG TIẾNG TOÀN BỘ KỊCH BẢN", variant="primary", size="lg")

                with gr.Column(scale=1):
                    with gr.Accordion("👥 Thiết lập Giọng & Seed Nhanh (M1-M10, F1-F10)", open=True):
                        with gr.Row():
                            btn_sync_tab1 = gr.Button("🔄 Nạp / Đồng Bộ Seed Mới Nhất (Đã Lưu Từ Tab 2)", variant="secondary", size="sm")
                        gr.Markdown("*(Đổi số Seed trực tiếp ở đây hoặc bên Tab 2, hệ thống sẽ tự động lưu và khóa giọng)*")
                        with gr.Tabs():
                            with gr.TabItem("👦 Nhân Vật Nam (M1 - M10)"):
                                with gr.Row():
                                    m1_cfg = gr.Textbox(label="M1 (Nam chính - Trầm ấm)", value=CHARACTER_DEFAULTS.get("M1", {}).get("instruct", "male, young adult, low pitch"), lines=1, scale=3)
                                    m1_seed = gr.Number(label="Seed M1", value=CHARACTER_DEFAULTS.get("M1", {}).get("seed", 15161), precision=0, scale=1)
                                with gr.Row():
                                    m2_cfg = gr.Textbox(label="M2 (Nam phụ / Trùm / Sếp)", value=CHARACTER_DEFAULTS.get("M2", {}).get("instruct", "male, middle-aged, very low pitch"), lines=1, scale=3)
                                    m2_seed = gr.Number(label="Seed M2", value=CHARACTER_DEFAULTS.get("M2", {}).get("seed", 1002), precision=0, scale=1)
                                with gr.Row():
                                    m3_cfg = gr.Textbox(label="M3 (Lão gia / Sư phụ)", value=CHARACTER_DEFAULTS.get("M3", {}).get("instruct", "male, elderly, low pitch"), lines=1, scale=3)
                                    m3_seed = gr.Number(label="Seed M3", value=CHARACTER_DEFAULTS.get("M3", {}).get("seed", 1003), precision=0, scale=1)
                                with gr.Row():
                                    m4_cfg = gr.Textbox(label="M4 (Nam thiếu niên / Thiếu gia)", value=CHARACTER_DEFAULTS.get("M4", {}).get("instruct", "male, teenager, moderate pitch"), lines=1, scale=3)
                                    m4_seed = gr.Number(label="Seed M4", value=CHARACTER_DEFAULTS.get("M4", {}).get("seed", 1004), precision=0, scale=1)
                                with gr.Row():
                                    m5_cfg = gr.Textbox(label="M5 (Giang hồ / Hổ báo)", value=CHARACTER_DEFAULTS.get("M5", {}).get("instruct", "male, middle-aged, low pitch"), lines=1, scale=3)
                                    m5_seed = gr.Number(label="Seed M5", value=CHARACTER_DEFAULTS.get("M5", {}).get("seed", 1005), precision=0, scale=1)
                                with gr.Row():
                                    m6_cfg = gr.Textbox(label="M6 (Thư sinh / Tri thức)", value=CHARACTER_DEFAULTS.get("M6", {}).get("instruct", "male, young adult, moderate pitch"), lines=1, scale=3)
                                    m6_seed = gr.Number(label="Seed M6", value=CHARACTER_DEFAULTS.get("M6", {}).get("seed", 1006), precision=0, scale=1)
                                with gr.Row():
                                    m7_cfg = gr.Textbox(label="M7 (Hài hước / Lém lỉnh)", value=CHARACTER_DEFAULTS.get("M7", {}).get("instruct", "male, young adult, energetic, high pitch"), lines=1, scale=3)
                                    m7_seed = gr.Number(label="Seed M7", value=CHARACTER_DEFAULTS.get("M7", {}).get("seed", 1007), precision=0, scale=1)
                                with gr.Row():
                                    m8_cfg = gr.Textbox(label="M8 (Chiến binh / Dũng tướng)", value=CHARACTER_DEFAULTS.get("M8", {}).get("instruct", "male, adult, strong, resonant"), lines=1, scale=3)
                                    m8_seed = gr.Number(label="Seed M8", value=CHARACTER_DEFAULTS.get("M8", {}).get("seed", 1008), precision=0, scale=1)
                                with gr.Row():
                                    m9_cfg = gr.Textbox(label="M9 (Quái kiệt / Dị nhân)", value=CHARACTER_DEFAULTS.get("M9", {}).get("instruct", "male, middle-aged, raspy, eccentric"), lines=1, scale=3)
                                    m9_seed = gr.Number(label="Seed M9", value=CHARACTER_DEFAULTS.get("M9", {}).get("seed", 1009), precision=0, scale=1)
                                with gr.Row():
                                    m10_cfg = gr.Textbox(label="M10 (Đạo sĩ / Cao nhân)", value=CHARACTER_DEFAULTS.get("M10", {}).get("instruct", "male, elderly, calm, airy"), lines=1, scale=3)
                                    m10_seed = gr.Number(label="Seed M10", value=CHARACTER_DEFAULTS.get("M10", {}).get("seed", 1010), precision=0, scale=1)

                            with gr.TabItem("👧 Nhân Vật Nữ (F1 - F10)"):
                                with gr.Row():
                                    f1_cfg = gr.Textbox(label="F1 (Nữ chính - Ngọt ngào)", value=CHARACTER_DEFAULTS.get("F1", {}).get("instruct", "female, young adult, high pitch"), lines=1, scale=3)
                                    f1_seed = gr.Number(label="Seed F1", value=CHARACTER_DEFAULTS.get("F1", {}).get("seed", 2001), precision=0, scale=1)
                                with gr.Row():
                                    f2_cfg = gr.Textbox(label="F2 (Nữ phụ / Mẹ / Phu nhân)", value=CHARACTER_DEFAULTS.get("F2", {}).get("instruct", "female, middle-aged, moderate pitch"), lines=1, scale=3)
                                    f2_seed = gr.Number(label="Seed F2", value=CHARACTER_DEFAULTS.get("F2", {}).get("seed", 2002), precision=0, scale=1)
                                with gr.Row():
                                    f3_cfg = gr.Textbox(label="F3 (Bé gái / Trẻ em)", value=CHARACTER_DEFAULTS.get("F3", {}).get("instruct", "female, child, high pitch"), lines=1, scale=3)
                                    f3_seed = gr.Number(label="Seed F3", value=CHARACTER_DEFAULTS.get("F3", {}).get("seed", 2003), precision=0, scale=1)
                                with gr.Row():
                                    f4_cfg = gr.Textbox(label="F4 (Nữ bí ẩn / Thì thầm)", value=CHARACTER_DEFAULTS.get("F4", {}).get("instruct", "female, young adult, whisper"), lines=1, scale=3)
                                    f4_seed = gr.Number(label="Seed F4", value=CHARACTER_DEFAULTS.get("F4", {}).get("seed", 2004), precision=0, scale=1)
                                with gr.Row():
                                    f5_cfg = gr.Textbox(label="F5 (Bà lão / Lão bà bà)", value=CHARACTER_DEFAULTS.get("F5", {}).get("instruct", "female, elderly, moderate pitch"), lines=1, scale=3)
                                    f5_seed = gr.Number(label="Seed F5", value=CHARACTER_DEFAULTS.get("F5", {}).get("seed", 2005), precision=0, scale=1)
                                with gr.Row():
                                    f6_cfg = gr.Textbox(label="F6 (Tiểu thư / Kiêu kỳ)", value=CHARACTER_DEFAULTS.get("F6", {}).get("instruct", "female, young adult, clear, arrogant"), lines=1, scale=3)
                                    f6_seed = gr.Number(label="Seed F6", value=CHARACTER_DEFAULTS.get("F6", {}).get("seed", 2006), precision=0, scale=1)
                                with gr.Row():
                                    f7_cfg = gr.Textbox(label="F7 (Nữ sát thủ / Lạnh lùng)", value=CHARACTER_DEFAULTS.get("F7", {}).get("instruct", "female, young adult, cold, low pitch"), lines=1, scale=3)
                                    f7_seed = gr.Number(label="Seed F7", value=CHARACTER_DEFAULTS.get("F7", {}).get("seed", 2007), precision=0, scale=1)
                                with gr.Row():
                                    f8_cfg = gr.Textbox(label="F8 (Nữ hiền dịu / Trầm tính)", value=CHARACTER_DEFAULTS.get("F8", {}).get("instruct", "female, adult, soft, gentle"), lines=1, scale=3)
                                    f8_seed = gr.Number(label="Seed F8", value=CHARACTER_DEFAULTS.get("F8", {}).get("seed", 2008), precision=0, scale=1)
                                with gr.Row():
                                    f9_cfg = gr.Textbox(label="F9 (Hoạt bát / Năng động)", value=CHARACTER_DEFAULTS.get("F9", {}).get("instruct", "female, teenager, lively, cheerful"), lines=1, scale=3)
                                    f9_seed = gr.Number(label="Seed F9", value=CHARACTER_DEFAULTS.get("F9", {}).get("seed", 2009), precision=0, scale=1)
                                with gr.Row():
                                    f10_cfg = gr.Textbox(label="F10 (Hoàng hậu / Nữ vương)", value=CHARACTER_DEFAULTS.get("F10", {}).get("instruct", "female, adult, majestic, noble"), lines=1, scale=3)
                                    f10_seed = gr.Number(label="Seed F10", value=CHARACTER_DEFAULTS.get("F10", {}).get("seed", 2010), precision=0, scale=1)

                        gr.Markdown("---")
                        gr.Markdown("🎙️ **Clone Nhái Giọng Riêng Nhanh (Tùy chọn tải file audio để gán giọng)**:")
                        with gr.Row():
                            m1_audio = gr.Audio(label="File mẫu giọng M1 (Nếu muốn clone nhanh)", type="filepath")
                            f1_audio = gr.Audio(label="File mẫu giọng F1 (Nếu muốn clone nhanh)", type="filepath")

                    out_audio = gr.Audio(label="🎧 File Âm Thanh Toàn Bộ Kịch Bản Đã Ghép", type="numpy")
                    out_file_down = gr.File(label="📥 Tải File Âm Thanh (.wav) Về Máy")
                    out_status = gr.Textbox(label="Nhật ký tiến trình từng câu", lines=6)

            all_seeds = [
                m1_seed, m2_seed, m3_seed, m4_seed, m5_seed, m6_seed, m7_seed, m8_seed, m9_seed, m10_seed,
                f1_seed, f2_seed, f3_seed, f4_seed, f5_seed, f6_seed, f7_seed, f8_seed, f9_seed, f10_seed
            ]
            all_cfgs = [
                m1_cfg, m2_cfg, m3_cfg, m4_cfg, m5_cfg, m6_cfg, m7_cfg, m8_cfg, m9_cfg, m10_cfg,
                f1_cfg, f2_cfg, f3_cfg, f4_cfg, f5_cfg, f6_cfg, f7_cfg, f8_cfg, f9_cfg, f10_cfg
            ]

            btn_sync_tab1.click(
                sync_tab1_seeds,
                outputs=all_seeds + all_cfgs
            )

            btn_generate.click(
                generate_review_dubbing,
                inputs=[
                    script_input, file_upload, lang_dd, lock_voice_box, align_box, gap_slider, speed_slider, step_choice,
                    *all_cfgs,
                    *all_seeds,
                    m1_audio, f1_audio
                ],
                outputs=[out_audio, out_status, out_file_down],
                api_name="generate_review_dubbing"
            )

        # TAB 2: QUẢN LÝ & KHÓA GIỌNG CHUẨN (VOICE CAST STUDIO)
        with gr.TabItem("👥 Quản Lý & Khóa Giọng Chuẩn (Voice Cast Studio)"):
            gr.Markdown("""
            ### 🔒 Khu Vực Thử Giọng & Cố Định Âm Sắc Cho Từng Nhân Vật
            > 🛡️ **NGUYÊN TẮC BẢO VỆ GIỌNG (TUYỆT ĐỐI KHÔNG TỰ ĐỘNG ĐỔI GIỌNG)**:
            > - **Hệ thống KHÔNG TỰ ĐỘNG ĐỔI giọng của nhân vật**: Mọi thao tác đổi Seed hay nghe thử đều là **NGHE THỬ TẠM THỜI**, tuyệt đối không ghi đè làm mất giọng đã lưu!
            > - **Chỉ lưu khi bạn chủ động bấm "💾 Áp Dụng & Lưu Làm Giọng Chuẩn Mới"**: Bạn có thể tự do thử hàng trăm seed ngẫu nhiên mà không lo bị thay đổi chất giọng gốc.
            > - **Nút Khôi Phục Giọng Gốc**: Nếu lỡ tay lưu nhầm, bấm nút **"🔄 Khôi Phục Lại Giọng Clone Gốc"** để lấy lại ngay giọng ban đầu từ code!
            """)
            with gr.Row():
                with gr.Column(scale=1):
                    with gr.Row():
                        char_lang_dd = gr.Dropdown(
                            label="🌐 Quốc Gia / Ngôn Ngữ Của Kho Giọng",
                            choices=SUPPORTED_LANGS,
                            value="Tiếng Việt (vi)",
                            scale=2
                        )
                        char_selector = gr.Dropdown(
                            label="Chọn Nhân Vật",
                            choices=list(CHARACTER_DEFAULTS.keys()),
                            value="M1",
                            scale=1
                        )
                    char_info_md = gr.Markdown("**M1** (VI): Nam trẻ trầm ấm, nam tính, phong thái anh hùng")

                    char_preview_text = gr.Textbox(
                        label="📝 Nội dung câu nói đọc thử",
                        value="Xin chào, đây là câu thoại thử nghiệm chất giọng chuẩn của nhân vật.",
                        lines=2
                    )

                    with gr.Row():
                        btn_play_current = gr.Button("🎧 Nghe Thử Giọng Hiện Tại Của Nhân Vật", variant="primary", size="lg")
                        btn_restore_clone = gr.Button("🔄 Khôi Phục Giọng Clone Gốc (Từ Code)", variant="secondary", size="lg")

                    gr.Markdown("---")
                    gr.Markdown("#### 🧪 Thử Nghiệm Giọng Mới (Tùy Chọn Đổi Seed Hoặc Instruct)")

                    char_seed_input = gr.Number(label="Số Seed Muốn Thử Nghiệm", value=15161, precision=0)
                    char_inst_input = gr.Textbox(label="Mô tả giọng (Instruct)", value="male, young adult, low pitch", lines=2)
                    char_custom_audio = gr.Audio(label="Hoặc Tải File Audio Người Thật Lên Để Clone Mới", type="filepath")

                    with gr.Row():
                        btn_test_voice = gr.Button("🔊 Nghe Thử Seed Này (KHÔNG Lưu)", variant="secondary", size="md")
                        btn_reroll_voice = gr.Button("🎲 Đổi Thử Seed Khác (KHÔNG Lưu)", variant="secondary", size="md")

                    with gr.Row():
                        btn_save_voice = gr.Button("💾 Áp Dụng & Lưu Làm Giọng Chuẩn Mới Cho Nhân Vật", variant="stop", size="lg")

                with gr.Column(scale=1):
                    char_preview_audio = gr.Audio(label="🎧 Trình Phát Âm Thanh Mẫu", type="filepath")
                    char_status_log = gr.Textbox(
                        label="Trạng thái lưu trữ giọng",
                        value="🛡️ Hệ thống đang bảo vệ 100% file giọng gốc. Mọi thao tác nghe thử sẽ không làm thay đổi giọng nhân vật.",
                        lines=5
                    )

                    with gr.Accordion("💾 Tải Riêng / Nạp Riêng File Giọng Cho Nhân Vật Này", open=True):
                        gr.Markdown("*(Dành cho ai muốn cất riêng file giọng của nhân vật này về máy tính cá nhân)*")
                        with gr.Row():
                            single_pt_file = gr.File(label="📥 File Giọng Riêng (.pt)")
                            single_wav_file = gr.File(label="🎧 File Audio Mẫu (.wav)")
                        with gr.Row():
                            btn_download_pt = gr.Button("📥 Lấy Link Tải File .pt", variant="secondary", size="sm")
                            btn_download_wav = gr.Button("🎧 Lấy Link Tải File .wav", variant="secondary", size="sm")
                        with gr.Row():
                            single_upload_input = gr.File(label="Nạp File Giọng (.pt hoặc .wav) Để Gán Riêng Cho Nhân Vật Này", file_types=[".pt", ".wav", ".mp3"])
                            btn_import_single = gr.Button("📤 Gán File Này Cho Nhân Vật", variant="primary", size="sm")

                    with gr.Accordion("📁 Tùy Chọn Nâng Cao: Sao Lưu Toàn Bộ Dự Án (Tất Cả Nhân Vật vào 1 File .zip)", open=False):
                        gr.Markdown("*(Chỉ dùng khi bạn muốn đóng gói tất cả các nhân vật M1-M10, F1-F10 mang sang máy khác)*")
                        with gr.Row():
                            btn_export_bank = gr.Button("💾 Đóng Gói & Tải Tất Cả Nhân Vật (.zip)", variant="secondary")
                            zip_download_file = gr.File(label="File ZIP Toàn Bộ Dự Án")
                        with gr.Row():
                            zip_upload_input = gr.File(label="Nạp File ZIP Toàn Bộ Dự Án", file_types=[".zip"])
                            btn_import_bank = gr.Button("📥 Khôi Phục Toàn Bộ", variant="secondary")
                        import_log = gr.Textbox(label="Nhật ký khôi phục toàn bộ", lines=2)

                    with gr.Accordion("📋 Lưu Vĩnh Viễn Vào Code Notebook (Dành cho Kaggle khi khởi động lại)", open=False):
                        gr.Markdown("*(Bấm nút dưới đây để lấy đoạn mã Python chứa toàn bộ Seed hoặc file giọng Clone .pt. Copy dán vào ô Code ở đầu ô [BƯỚC 2/2] là vĩnh viễn không bao giờ bị mất!)*")
                        with gr.Row():
                            btn_export_code = gr.Button("📋 Xuất Mã Cấu Hình Seed (CHARACTER_DEFAULTS)", variant="secondary", size="sm")
                            btn_export_b64 = gr.Button("🧬 Xuất Mã Nhúng Base64 File Giọng Clone (.pt) Nhân Vật Này", variant="secondary", size="sm")
                        code_output_box = gr.Textbox(label="Mã Code Python (Copy dán vào Cell 2 trên Kaggle)", lines=6)
                        btn_export_code.click(export_character_defaults_code, outputs=[code_output_box])
                        btn_export_b64.click(export_char_base64_code, inputs=[char_selector], outputs=[code_output_box])

            char_selector.change(
                on_select_character,
                inputs=[char_selector, char_lang_dd],
                outputs=[char_seed_input, char_inst_input, char_info_md, char_preview_audio, char_status_log, single_pt_file, single_wav_file]
            )
            char_lang_dd.change(
                on_select_character,
                inputs=[char_selector, char_lang_dd],
                outputs=[char_seed_input, char_inst_input, char_info_md, char_preview_audio, char_status_log, single_pt_file, single_wav_file]
            )

            # Nút 1: Nghe thử đúng giọng chuẩn hiện tại (Bảo toàn 100% không đổi giọng)
            btn_play_current.click(
                play_character_current_voice,
                inputs=[char_selector, char_preview_text, char_lang_dd],
                outputs=[char_preview_audio, char_status_log, single_pt_file, single_wav_file]
            )

            # Nút 2: Khôi phục giọng clone gốc từ code EMBEDDED_CLONE_VOICES
            btn_restore_clone.click(
                restore_original_clone_voice,
                inputs=[char_selector, char_preview_text, char_lang_dd],
                outputs=[char_seed_input, char_preview_audio, char_status_log, single_pt_file, single_wav_file]
            )

            # Nút 3: Nghe thử với Seed/Instruct mới (Chỉ nghe thử tạm thời, KHÔNG lưu)
            btn_test_voice.click(
                preview_temporary_voice,
                inputs=[char_selector, char_seed_input, char_inst_input, char_preview_text, char_custom_audio, char_lang_dd],
                outputs=[char_preview_audio, char_status_log]
            )

            # Nút 4: Đổi thử Seed khác ngẫu nhiên (Chỉ đổi số để nghe thử, KHÔNG lưu)
            def reroll_seed_only(tag, instruct, preview_text, audio_file, lang):
                import random
                new_seed = random.randint(1000, 99999)
                wav_path, msg = preview_temporary_voice(tag, new_seed, instruct, preview_text, audio_file, lang)
                return new_seed, wav_path, f"🎲 Đã đổi thử sang Seed {new_seed}!\n" + (msg if msg else "")

            btn_reroll_voice.click(
                reroll_seed_only,
                inputs=[char_selector, char_inst_input, char_preview_text, char_custom_audio, char_lang_dd],
                outputs=[char_seed_input, char_preview_audio, char_status_log]
            )

            # Nút 5: Xác nhận lưu giọng mới (Chỉ khi bấm nút này mới lưu)
            btn_save_voice.click(
                confirm_save_character_voice,
                inputs=[char_selector, char_seed_input, char_inst_input, char_custom_audio, char_lang_dd],
                outputs=[char_preview_audio, char_status_log, single_pt_file, single_wav_file]
            )

            btn_download_pt.click(get_single_char_pt, inputs=[char_selector, char_lang_dd], outputs=[single_pt_file])
            btn_download_wav.click(get_single_char_wav, inputs=[char_selector, char_lang_dd], outputs=[single_wav_file])
            btn_import_single.click(import_single_char_file, inputs=[char_selector, single_upload_input, char_lang_dd], outputs=[char_preview_audio, char_status_log, single_pt_file, single_wav_file])

            btn_export_bank.click(export_voice_bank_zip, outputs=[zip_download_file])
            btn_import_bank.click(import_voice_bank_zip, inputs=[zip_upload_input], outputs=[import_log])

        # TAB 3: VOICE CLONE ĐƠN LẺ (CHUẨN GRADIO, KHÔNG LỒNG BLOCKS)
        with gr.TabItem("🎙️ Voice Clone Đơn Lẻ (Nhái 1 Giọng)"):
            gr.Markdown("""
            ### 🎙️ Nhái Giọng Bất Kỳ Từ 1 File Âm Thanh Mẫu (Voice Clone)
            - Tải file âm thanh mẫu (3 - 10 giây giọng người thật) lên.
            - Nhập câu thoại muốn đọc -> Bấm **"Sinh Giọng Nhái"**.
            - Sau khi ưng ý, bạn có thể **Tải file giọng mẫu (.pt) về máy** (y như các file `F2.pt`, `F3.pt`...) hoặc **Gán thẳng làm giọng chuẩn cho nhân vật (M1, F1...)** để lồng tiếng phim!
            """)
            with gr.Row():
                with gr.Column():
                    single_text = gr.Textbox(
                        label="Văn bản cần đọc thử",
                        placeholder="Nhập văn bản cần đọc...",
                        lines=3,
                        value="Xin chào, đây là bản thử nghiệm nhái giọng nói đơn lẻ bằng công nghệ trí tuệ nhân tạo OmniVoice."
                    )
                    single_ref_audio = gr.Audio(label="File âm thanh mẫu người thật (3 - 10 giây)", type="filepath")
                    gr.Markdown("*(Hỗ trợ mọi định dạng: .mp3, .wav, .m4a, .aac, .webm... Khuyến nghị: 3 - 10 giây rõ tiếng)*")
                    single_ref_text = gr.Textbox(label="Nội dung lời nói trong file mẫu (Tùy chọn - Giúp đọc chuẩn hơn)", lines=2)
                    with gr.Row():
                        single_speed = gr.Slider(0.5, 1.5, value=1.0, step=0.05, label="Tốc độ đọc")
                        single_lang = gr.Dropdown(
                            label="Ngôn ngữ",
                            choices=SUPPORTED_LANGS + ["Auto"],
                            value="Tiếng Việt (vi)"
                        )
                    btn_single_clone = gr.Button("🚀 Sinh Giọng Nhái (Clone Voice)", variant="primary", size="lg")
                with gr.Column():
                    single_out_audio = gr.Audio(label="🎧 Kết quả âm thanh đã nhái", type="filepath")
                    single_status = gr.Textbox(label="Trạng thái", lines=3)

                    with gr.Accordion("💾 Lưu & Xuất Voice Mẫu Để Dùng Lại (File .pt)", open=True):
                        gr.Markdown("#### 📥 Cách 1: Tải File Giọng Mẫu (.pt) Về Máy Tính (Y hệt như file F2.pt, F3.pt...)")
                        with gr.Row():
                            cloned_pt_file = gr.File(label="📥 Tải File Giọng Mẫu (.pt) Về Máy")
                            cloned_wav_file = gr.File(label="🎧 Tải File Âm Thanh Kết Quả (.wav)")

                        gr.Markdown("#### 🎯 Cách 2: Gán Thẳng Làm Giọng Chuẩn Cho Nhân Vật Trong Phim (Tách Biệt Theo Quốc Gia)")
                        with gr.Row():
                            target_lang_dd = gr.Dropdown(
                                label="Quốc Gia / Ngôn Ngữ:",
                                choices=SUPPORTED_LANGS,
                                value="Tiếng Việt (vi)",
                                scale=2
                            )
                            target_char_dd = gr.Dropdown(
                                label="Chọn nhân vật:",
                                choices=[f"M{i}" for i in range(1, 11)] + [f"F{i}" for i in range(1, 11)],
                                value="M1",
                                scale=1
                            )
                            btn_assign_char = gr.Button("🎯 Gán Cho Nhân Vật", variant="primary", scale=2)
                        assign_status_box = gr.Textbox(label="Kết quả gán nhân vật", lines=2)

                        gr.Markdown("#### 🧬 Cách 3: Nhúng Thẳng Vào Code Notebook (Lưu Vĩnh Viễn Không Sợ Kaggle Xóa)")
                        with gr.Row():
                            btn_export_tab3_b64 = gr.Button("📋 Lấy Mã Nhúng Base64 Của Giọng Nhái Này", variant="secondary")
                        b64_output_box = gr.Textbox(label="Mã Code Python (Copy dán vào biến EMBEDDED_CLONE_VOICES ở đầu Cell 2)", lines=4)
                        btn_export_tab3_b64.click(export_last_cloned_base64_code, inputs=[target_char_dd], outputs=[b64_output_box])

            btn_single_clone.click(
                process_single_clone,
                inputs=[single_text, single_ref_audio, single_ref_text, single_speed, single_lang],
                outputs=[single_out_audio, single_status, cloned_pt_file, cloned_wav_file]
            )

            btn_assign_char.click(
                assign_cloned_voice_to_character,
                inputs=[target_char_dd, target_lang_dd],
                outputs=[assign_status_box]
            )

# Danh sách đường dẫn được cấp phép phục vụ file âm thanh (Chống lỗi InvalidPathError trên Colab/Kaggle)
allowed_dirs = list(set([
    os.getcwd(),
    os.path.abspath("."),
    tempfile.gettempdir(),
    BASE_DIR,
    CACHE_DIR,
    "/content",
    "/kaggle/working"
]))
allowed_dirs = [p for p in allowed_dirs if os.path.exists(p)]

# 1. Dọn dẹp log tải mô hình dài dòng TRƯỚC KHI mở giao diện để giữ chỗ cho Web UI sạch đẹp
try:
    from IPython.display import clear_output
    clear_output(wait=True)
except Exception:
    pass

print("="*65)
print("="*65)
print("🚀 [3/3] ĐANG KẾT NỐI ĐƯỜNG TRUYỀN CỐ ĐỊNH & KHỞI CHẠY WEB UI...", flush=True)
print("="*65)

# 1. KẾT NỐI ĐƯỜNG TRUYỀN CỐ ĐỊNH NGROK TRƯỚC (BỎ QUA HOÀN TOÀN LỖI CỦA GRADIO SHARE!)
share_url = None
try:
    from pyngrok import ngrok
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=False)
    from pyngrok import ngrok

try:
    ngrok.kill()
    ngrok.set_auth_token(str(NGROK_AUTHTOKEN).strip())
    t_opts = {"addr": 7860, "proto": "http"}
    if NGROK_DOMAIN and str(NGROK_DOMAIN).strip():
        t_opts["domain"] = str(NGROK_DOMAIN).strip()
    tunnel = ngrok.connect(**t_opts)
    share_url = tunnel.public_url.replace("http://", "https://")
    print("="*65)
    print(f"🎉 ĐÃ KẾT NỐI ĐƯỜNG TRUYỀN NGROK CỐ ĐỊNH VĨNH VIỄN: {share_url}")
    print("="*65)
except Exception as ngrok_err:
    print(f"⚠️ Ngrok: {ngrok_err}. Chuyển sang phương án Cloudflare...")
    try:
        import subprocess
        cf_bin = "/usr/local/bin/cloudflared"
        if not os.path.exists(cf_bin):
            subprocess.run(["wget", "-q", "-nc", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", cf_bin], check=False)
            subprocess.run(["chmod", "+x", cf_bin], check=False)
        if os.path.exists(cf_bin):
            cf_log = "/tmp/cloudflared.log"
            if os.path.exists(cf_log):
                try: os.remove(cf_log)
                except: pass
            subprocess.Popen([cf_bin, "tunnel", "--url", "http://127.0.0.1:7860"], stdout=subprocess.DEVNULL, stderr=open(cf_log, "w"))
            for _ in range(15):
                time.sleep(1)
                if os.path.exists(cf_log):
                    with open(cf_log, "r", encoding="utf-8", errors="ignore") as lf:
                        c_text = lf.read()
                    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', c_text)
                    if m:
                        share_url = m.group(0)
                        print(f"🎉 ĐÃ MỞ CLOUDFLARE TUNNEL TỐC ĐỘ CAO: {share_url}")
                        break
    except Exception as cf_e:
        print(f"⚠️ Cloudflare: {cf_e}")

# 2. Khởi chạy Gradio Server với share=False (ĐỂ KHÔNG BAO GIỜ BỊ LỖI STATUS.GRADIO.APP NỮA!)
launch_result = custom_app.queue().launch(
    share=False,
    server_name="0.0.0.0",
    server_port=7860,
    inline=False,
    prevent_thread_lock=True,
    show_error=True,
    allowed_paths=allowed_dirs
)

# 3. Hiển thị banner điều hướng nổi bật
try:
    from IPython.display import display, HTML
    if share_url:
        display(HTML(f"""
        <div style="background: linear-gradient(135deg, #0d1b2a 0%, #1b263b 50%, #415a77 100%); padding: 22px; border-radius: 18px; text-align: center; box-shadow: 0 8px 30px rgba(0,0,0,0.5); margin: 15px auto; max-width: 820px; border: 2px solid #00e5ff;">
            <h2 style="color: #00e5ff; margin: 0 0 8px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 24px;">
                🎉 OMNIVOICE STUDIO ĐÃ SẴN SÀNG (ĐƯỜNG TRUYỀN CỐ ĐỊNH)!
            </h2>
            <p style="color: #e0e1dd; font-size: 15px; margin: 0 0 18px 0;">
                Bấm nút bên dưới để mở giao diện Web UI hoặc copy link dán vào Tool Review Video:
            </p>
            <a href="{share_url}" target="_blank" style="background: linear-gradient(135deg, #FF416C 0%, #FF4B2B 100%); color: #ffffff; padding: 15px 38px; text-decoration: none; border-radius: 50px; font-size: 20px; font-weight: 800; display: inline-block; box-shadow: 0 5px 20px rgba(255, 75, 43, 0.5); letter-spacing: 0.5px; text-transform: uppercase;">
                🚀 BẤM VÀO ĐÂY ĐỂ MỞ WEB UI (CỐ ĐỊNH VĨNH VIỄN)
            </a>
            <div style="margin-top: 14px; font-size: 14px; color: #778da9;">
                <span>Link kết nối Tool: </span>
                <a href="{share_url}" target="_blank" style="color: #64ffda; font-weight: bold; font-size: 16px; text-decoration: underline;">{share_url}</a>
            </div>
        </div>
        """))
    else:
        display(HTML("""
        <div style="background: #2b1b17; border: 2px solid #ff9800; padding: 18px; border-radius: 14px; text-align: center; margin: 15px auto; max-width: 780px;">
            <h3 style="color: #ff9800; margin: 0 0 8px 0;">👇 GIAO DIỆN WEB UI ĐANG CHẠY TRỰC TIẾP Ở KHUNG BÊN DƯỚI!</h3>
        </div>
        """))
except Exception:
    pass

print("\n" + "="*65)
if share_url:
    print(f"🚀 LINK DÁN VÀO TOOL REVIEW: {share_url}")
else:
    print("ℹ️ Đang chạy trực tiếp.")
print("="*65 + "\n")

# 4. Giữ tiến trình luôn hoạt động
try:
    if hasattr(custom_app, "block_thread"):
        custom_app.block_thread()
    else:
        while True:
            time.sleep(1)
except KeyboardInterrupt:
    print("Đã dừng server Gradio.")
